# Tilt Correction

Almost every step of a reduction assumes that a **column of pixels is a single wavelength**.
Chapter 7 will subtract sky measured 60 rows away from the source and apply it to the source's own
row. Chapters 10 and 11 will fit a wavelength solution to arc lines measured in one row and apply
it to a spectrum extracted from another. Both assume that moving along the slit does not change
which wavelength lands in a given column.

On these frames that assumption is false, and not by a little. Chapters 1 and 5 showed the arc
lines leaning; this chapter measures the lean, models it, and removes it. It does so **before**
anything else is measured from the frames, which is the order a real reduction has to use.

The cause is optical. A spectrograph images the slit through a dispersing element onto a detector,
and no real optical train does that perfectly: the monochromatic image of a long slit ends up
slightly curved and tilted rather than exactly parallel to the detector columns. The consequence
is that **wavelength varies along the cross-dispersion axis**. Lines that should be vertical lean
and bow, increasingly so toward the edges of the slit.

`specreduce.tilt_correction.TiltCorrection` handles this in four steps:

1. **Find arc lines** at a set of sample positions along the slit, building a map of where each
   line actually falls as a function of row.
2. **Fit a 2D polynomial** describing the transformation between a rectified coordinate space and
   detector space.
3. **Check** the fit against the measured line positions.
4. **Rectify**: resample each frame onto the corrected grid, so that columns become wavelengths
   again. This last step is optional; see the note below.

This matters for real science. The transmission spectrum published from these observations
([Parviainen et al. 2016](https://ui.adsabs.harvard.edu/abs/2016A%26A...585A.114P/abstract))
carried a Rayleigh-like slope that the paper left unexplained and that, as Chapter 1 recounts,
came in hindsight from an insufficient tilt correction: exactly the kind of wavelength-dependent
systematic that can imitate a planetary atmosphere in transmission spectroscopy.

:::{note}
**This step is optional, in two senses.** Whether you need it at all depends on your
instrument: these OSIRIS frames have an unusually strong tilt, which makes them a good teaching
case but not a typical one, and the section "Do you need this step at all?" shows how to decide
whether your own data needs the correction. And even when the tilt does need modeling,
resampling the frames is a choice rather than a requirement. The fitted `TiltSolution` is a
coordinate transformation in its own right, exposed as a GWCS and written to ASDF (see "Saving
the solution" below), and it can be handed to a reduction that works on the unrectified detector
frame, for example one that extracts along curves of constant wavelength or forward-models the
2D frame, without ever resampling the data. This book rectifies, because the background
subtraction and extraction that follow assume a column is a single wavelength; Chapter 12 then
shows the other route, a 2D wavelength solution that combines the tilt and dispersion models and
works on the unrectified frame.
:::

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from specreduce.tilt_correction import TiltCorrection
from specreduce.tracing import FlatTrace

from common import TILT_SOLUTION_FILE, read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape

## Seeing the tilt

Arc lamps make the distortion obvious, because an arc frame is nothing *but* emission lines. Each
line is a monochromatic image of the slit, so it should be a perfectly vertical bar. Here is the
Ne frame, stretched so the lines are easy to follow.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
ne = arcs[lamps.index("Ne")].data

ax.imshow(
    ne,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    interpolation="bicubic",
    norm=simple_norm(ne, stretch="log", vmin=0, vmax=150_000),
)
ax.set(
    xlim=(690, 800),
    ylim=(0, 371),
    xlabel="Dispersion [pix]",
    ylabel="Cross-dispersion [pix]",
)
# Ticks on the top spine as well, so the lean can be read off at both ends of the slit.
ax.tick_params(axis="x", top=True, labeltop=True);

The lines lean visibly. To put a number on it, we take the intensity-weighted centroid of one
bright Ne line in successive 20-row bands along the slit. Between one end of the illuminated slit and the
other the line moves by five and a half pixels. The line itself is only about 2.5 pixels wide, so
the same wavelength shifts by more than twice the width of the feature. Nothing that assumes
constant wavelength down a column can survive that.

## Setting up `TiltCorrection`

`TiltCorrection` needs the arc frames and a **reference position**. The reference row defines the
rectified space: after correction, every row will have the wavelength scale that this row had
originally. It should therefore sit close to where the science spectrum falls, so that the
wavelength solution fitted later applies to the rows you actually extract.

Chapter 4 put the source at row 135, so we use that as the reference. A `FlatTrace` is the
simplest way to express it. This is a *reference row for the tilt fit*, not a trace of any source, so it does
not need to follow the science target's curvature.

In [ ]:
reference = FlatTrace(arcs[0], 135)

tilt = TiltCorrection(arcs, reference, n_cdisp_samples=7)

print(f"reference pixel (cross-disp, disp): {tilt.ref_pixel}")
print(f"cross-dispersion samples:           {tilt.cd_samples}")
print(f"arc frames:                         {tilt.nframes}")

Two arguments shape the sampling. `n_cdisp_samples` sets how many rows are sampled across the
slit; the seven chosen rows are listed above. `cdisp_sample_lims` bounds the region they are drawn
from, and is left at its default, the whole frame, because the loader has already trimmed the
frame to `TRIMSEC` (Chapter 2). On an untrimmed frame it is how you keep the samples out of the
prescan and the unilluminated strip below the slit, where there are no arc lines to find.

We pass all three lamps together. The tilt is a property of the **optics**, not of any
particular lamp, so every arc frame measures the same distortion; using all three
provides more lines and better coverage along the dispersion axis.

## Finding the arc lines

`find_arc_lines` detects emission peaks at the reference row and at each cross-dispersion sample,
using `specutils`' threshold-based line finder. Two parameters control the detection: `fwhm`, the
expected line width in pixels, and `noise_factor`, how far above the noise a peak must rise to
count.

The threshold is applied to the flux itself, not to the flux above the continuum, so it assumes a
row whose baseline is zero. An arc row is not that: scattered light puts a pedestal of 100 to
180 e- under these lines, against a per-pixel uncertainty of 16 to 18 e-, so the baseline alone
stands at 6 to 10 sigma before a single line is considered. At `noise_factor=10` the blue end of
the row sits just under the threshold, and wherever noise carries a run of baseline pixels over
it, the run registers as a line. `find_arc_lines` therefore estimates the baseline of every row
with a sigma-clipped median, which the emission lines cannot bias upward, and subtracts it before
thresholding. That is the default (`subtract_baseline=True`); `baseline_window` switches to a
running estimate for a background that varies along the row, and `subtract_baseline=False` is for
arc frames that were background-subtracted before they got here.

We use `noise_factor=25`, a generous margin. These arcs are bright enough that the
margin costs only the faintest detections, and what it buys is a clean sample: every remaining
detection is a well-measured line, and the fit statistics below are not diluted by marginal ones.

In [ ]:
tilt.find_arc_lines(fwhm=2.5, noise_factor=25)

print(f"{'lamp':>6}  {'lines at reference row':>23}  {'detector-space samples':>23}")
for lamp, reference_lines, samples in zip(lamps, tilt._lines_ref, tilt._samples_det_x):
    print(f"{lamp:>6}  {len(reference_lines):>23}  {len(samples):>23}")

The `fwhm=2.5` is the arc line width measured in [Chapter 5](05-arc-lamps.ipynb). The arcs were
taken through the narrow 1.23-arcsecond slit, so their lines are sharp, unlike the far broader sky
lines in the 40-arcsecond science exposure.

Two sets of points come out of this. The **reference lines** are those found in row 135; they
define the rectified coordinate system. The **detector-space samples** are every line found at
every sampled row, between 126 and 251 per lamp here, and they describe where those wavelengths
actually land. Because a faint line may be detected at some rows and missed at others, the two
sets are matched with a kd-tree rather than assumed to correspond one-to-one.

Not every detection survives. Each one is refined with a Gaussian fit, and on a broad, faint bump
that barely clears the threshold that fit can run away, leaving a centroid far outside the
frame. `find_arc_lines` discards any centroid that is not finite or falls off the frame along the
dispersion axis before it reaches the fit. At `noise_factor=10` that guard removes about twenty
detections on these frames; at 25 nothing marginal enough to fail is detected in the first place.

In [ ]:
fig, axs = plt.subplots(3, figsize=(8, 5.4), sharex="all", constrained_layout=True)

for ax, frame, lamp, detector_x, detector_y, reference_lines in zip(
    axs, arcs, lamps, tilt._samples_det_x, tilt._samples_det_y, tilt._lines_ref
):
    ax.imshow(
        frame.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(frame.data, stretch="log", vmin=0, vmax=250_000),
    )
    ax.plot(detector_x, detector_y, "k.", ms=2, label="detector-space samples")
    ax.plot(
        reference_lines,
        np.full_like(reference_lines, tilt.ref_pixel[0]),
        "r.",
        ms=3,
        label="reference row",
    )
    ax.text(0.01, 0.9, lamp, transform=ax.transAxes, va="top", size=8)
    ax.set_ylabel("Cross-disp. [pix]")

axs[-1].legend(frameon=False, loc="lower left", fontsize=7, ncols=2)
axs[-1].set_xlabel("Dispersion [pix]");

The black points trace out the tilt directly: each near-vertical chain of dots is one line, and
the chains lean consistently. The red points along the middle of each panel are the reference
lines. Notice that the three lamps populate different parts of the spectrum, HgAr toward the
blue and Xe toward the red, which is why the book carries all three.

## Fitting the transformation

`fit` finds the 2D polynomial mapping rectified coordinates to detector coordinates. It works in
two stages: a robust kd-tree-based minimization to get close, then a least-squares refinement on
matched line pairs. `refine_fit` runs that second stage again with a tighter
`match_distance_bound`, which is worth doing because the first pass has to be generous enough to
find matches at all, and generosity has a cost that the next paragraphs quantify.

In [ ]:
tilt.fit(degree=4)
tilt.refine_fit(degree=4, match_distance_bound=1.0)

`match_distance_bound` is the one setting in this step that deserves thought. After the first pass,
`refine_fit` transforms every reference line to detector space, pairs it with the nearest detected
sample, keeps the pairs closer than the bound, and refits the polynomial to them by least squares.
The bound therefore decides which points the fit sees, and it has to sit between two scales. Below
it is the centroiding scatter of the lines, about 0.04 pixels here; above it is the spacing of
neighboring arc lines, which on these frames gets down to 3 to 5 pixels in the dense parts of the
Ne and Xe spectra, and closer still where lines blend.

A bound of several pixels breaks the second scale. At 5 pixels the matcher keeps about 20 pairs in
which a line detected at one row has been paired with a *different* line at another; the rms of
the matched residuals rises from 0.04 to 0.55 pixels, and the refit solution shifts by up to 2.7
pixels across the frame. That rms is not a worse fit to the same points, it is a different sample,
and it is the mismatches that dominate it. Anything from a quarter of a pixel to 2 pixels, on the
other hand, gives solutions that agree with one another to better than a quarter of a pixel
everywhere on the detector, and all of them rectify the frame.

So the choice is not delicate, as long as it stays well inside the line spacing. We use
**1.0 pixel**: twenty-five times the centroiding scatter, so no real pair is lost because the
first-pass solution was slightly off, and a third of the closest line spacing, so no line is paired
with its neighbor. Among the pairs it keeps, the model reproduces the measured positions to about
**0.04 pixels**. But an internal fit statistic is never the final word, for a reason the next
section spells out.

`plot_fit_quality` shows the matched lines on the detector, with marker size proportional to the
residual, and the residuals against each axis in the margins. It uses a 5-pixel match bound by
default, so the few large markers are exactly the spurious pairs just discussed.

In [ ]:
tilt.plot_fit_quality(figsize=(8, 5), rlim=(-0.5, 0.5))

The transformation can also be viewed directly in detector space. `plot_wavelength_contours`
overlays curves of constant rectified wavelength, that is, the lines that *should* have been
vertical.

In [ ]:
fig, axs = plt.subplots(2, figsize=(8, 4.2), sharex="all", constrained_layout=True)

for ax, frame, label in ((axs[0], arcs[1], "Ne arc"), (axs[1], obj, "science")):
    ax.imshow(
        frame.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(frame.data, stretch="log", vmin=0, vmax=250_000),
    )
    tilt.plot_wavelength_contours(ax=ax)
    ax.text(0.01, 0.92, label, transform=ax.transAxes, va="top", size=8)
    ax.set(xlim=(0, nx - 1), ylabel="Cross-disp. [pix]")

axs[-1].set_xlabel("Dispersion [pix]");

The contours bow away from vertical, by several pixels at the top and bottom of the slit. They lie
on the arc lines in the upper panel, which is the visual confirmation that the model has captured
the real distortion. The same contours drawn on the science frame show the distortion that
was silently present when Chapter 4 traced it.

## Rectifying the frames

`solution.resample` applies the transformation, rebinning each frame onto the rectified grid with
an exact, flux-conserving algorithm.

In [ ]:
rectified = {
    label: tilt.solution.resample(frame)
    for label, frame in zip(lamps + ("science",), list(arcs) + [obj])
}

for label, frame in (("Ne", arcs[1]), ("science", obj)):
    before = frame.data.sum()
    after = rectified[label].data.sum()
    print(
        f"{label:>8}: flux before {before:.6e}, after {after:.6e}, "
        f"ratio {after / before:.6f}"
    )

science = rectified["science"]
print(f"\nresample returns: {type(science).__name__}, unit {science.unit}")
print(f"  uncertainty:    {type(science.uncertainty).__name__}")
print(f"  mask:           {science.mask.sum()} pixels masked")
print(f"  metadata:       {dict(science.meta)}")

Flux is conserved to better than one part in two thousand, the small loss being light that falls
outside the rectified grid at the frame edges.

Side by side, the Ne frame before and after rectification:

In [ ]:
fig, axs = plt.subplots(
    2, figsize=(8, 4.4), sharex="all", sharey="all", constrained_layout=True
)

for ax, image, label in (
    (axs[0], arcs[1].data, "Ne, raw"),
    (axs[1], rectified["Ne"].data, "Ne, rectified"),
):
    ax.imshow(
        image,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        interpolation="bicubic",
        norm=simple_norm(image, stretch="log", vmin=0, vmax=150_000),
    )
    ax.text(
        0.01,
        0.93,
        label,
        transform=ax.transAxes,
        va="top",
        size=8,
        bbox=dict(fc="white", ec="none", alpha=0.7, pad=1.5),
    )
    ax.set_ylabel("Cross-disp. [pix]")

axs[-1].set(xlim=(690, 800), ylim=(0, 371), xlabel="Dispersion [pix]");

Tracking the centroid of the bright Ne line from "Seeing the tilt" through the rectified frame
gives a drift of well under 0.1 pixels along the slit, down from 5.6, a factor of more than eighty
and comfortably below the point where it could matter for anything downstream. The same
wavelength now lands in the same column whichever row you look at, which is precisely the property
every later step assumes.

`resample` carries the whole `NDData` across, not just the flux: the variance is propagated
through the resampling, the mask marks any output bin that could not be filled, and the metadata,
including the read noise `read_file` measured, travels with it. That matters immediately, because
`Background` (Chapter 7) needs an uncertainty to propagate and `HorneExtract` (Chapter 8) refuses
to run without one. The rectified frame is ready to use as it stands.

:::{warning}
**A resampled variance is not the variance of an independent measurement.** Rectification splits
each detector pixel between the two output bins it straddles, and a bin that receives a fraction
$f$ of a pixel holding $N$ electrons receives $fN$ of the flux but only $f^2N$ of the variance. The
propagated variance is therefore *sub-Poisson*, about 40 percent below the independent-pixel model
on these frames. That is the correct diagonal of the covariance, but no per-pixel variance can
express the off-diagonal: neighboring columns of a rectified frame now share noise. Noise
estimators based on pixel-to-pixel differences read low on a resampled frame for that reason, and
Chapter 9 meets a consequence of the sub-Poisson diagonal: a detection threshold set in units of
the uncertainty is lower after rectification than before. Estimate your noise before resampling where you can, and treat per-pixel errors
on a rectified frame as a lower bound.
:::

## Choosing the parameters

Three settings shape the fit.

`degree` is the order of the 2D polynomial. Optical distortion is smooth, so low orders suffice:
3 is the minimum `fit` accepts, because it seeds its refinement from an internal degree-3 model,
and 4 is a comfortable default with a little headroom. Higher orders buy nothing on a smooth
distortion and can oscillate between the sampled rows.

`n_cdisp_samples` sets how many rows along the slit the lines are measured in, and it must stay
comfortably above `degree`. A polynomial of degree *d* in the cross-dispersion coordinate needs
more than *d* + 1 sample rows to be determined. With fewer, the fit passes exactly through its
constraints, reports a tiny residual, and does whatever it likes between and beyond them,
typically mapping the unsampled rows far off the detector so that `resample` finds nothing to put
there. **A low residual on an underdetermined fit is a symptom, not a reassurance.** Sample the
whole slit, use more rows than the degree strictly needs, and judge the result on the rectified
frame: the flux should still be there, and a given line should land in the same column at every
row. Beyond that, extra samples cost line-finding time and change nothing.

`arc_frames` accepts one frame or several. One lamp is enough in principle, because the
distortion belongs to the optics and every lamp measures the same thing; extra lamps buy coverage
and redundancy, more lines spread over the full dispersion range rather than clustered where one
lamp happens to emit. The fit residual is not the way to compare them: it measures the centroiding
scatter of whichever lines a lamp offers, so a lamp with a few bright, isolated lines can post a
lower rms than three lamps together while constraining the transformation less well. Single
versus multiple arcs is a parameter choice, not a different workflow.

## Do you need this step at all?

Everything above was worth doing because the tilt on these frames is severe. That is a property of
this spectrograph, not of spectroscopy, and it is worth being explicit that **tilt correction is
an optional step**. A well-aligned instrument can show a fraction of a pixel of drift across the
whole slit, in which case rectifying gains nothing and costs a resampling.

The test is cheap. Take the centroid of one arc line in successive bands of rows along the slit,
as was done for the five and a half pixels quoted at the start of the chapter, and divide the
total drift by the width of the line you measured it in. Here that is 5.6 pixels of drift against
an instrumental width of 2.5 pixels, a ratio of about 2.2.

That ratio is the number to judge. As a rule of thumb:

| drift / line width | what it means |
|---|---|
| well below 0.5 | the distortion is smaller than your centroiding error. Skip this chapter. |
| around 0.5 to 1 | tolerable for wavelength calibration, but it will show up in sky subtraction if the science slit is narrow enough to give sharp sky lines. |
| above 1 | correct it. A wavelength is landing in a different column at different points along the slit, and nothing downstream can recover from that. |

These frames, at **2.2**, are well into the third case.

Two things make that judgment instrument-specific rather than universal. The drift depends on the
optical design, and the line width depends on the slit, since a narrower slit gives sharper lines and so
makes the *same* physical distortion matter more. It is the ratio that decides, not either number
alone.

:::{note}
**If you skip this step, nothing downstream changes except the numbers.** The chapters that follow
call `Background`, `BoxcarExtract`, `HorneExtract` and `WavelengthCalibration1D` on the rectified
frames, but those functions neither know nor care whether a frame has been rectified. Substituting
the raw `obj` and `arcs` works exactly as well, and is the right thing to do if your instrument
does not need the correction.
:::


## Saving the solution

A tilt solution is a calibration product. It depends on the arc frames and on the instrument's
optics, not on any science exposure, so it is fitted once and applied to every frame taken in the
same configuration, which in a real program can mean hundreds of exposures across a night.
Refitting it for each one would be wasteful and, worse, would let the solution drift between
frames that ought to share it.

`TiltSolution.to_asdf` writes the solution to [ASDF](https://asdf-standard.readthedocs.io/), the
format Astropy uses for structured scientific data, and `TiltSolution.from_asdf` reads it back.
The file holds the transformation as an **analytic model**, a few kilobytes of polynomial
coefficients rather than a resampled image, so what comes back is the same polynomial to the last
bit, and resampling with the restored solution reproduces the original flux and variance exactly.

In [ ]:
tilt.solution.to_asdf(TILT_SOLUTION_FILE)

Every chapter after this one starts by reading it back:

In [ ]:
from specreduce.tilt_solution import TiltSolution
from common import TILT_SOLUTION_FILE

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

The model is stored as a [GWCS](https://gwcs.readthedocs.io/) object, and the same object is
available live as `TiltSolution.gwcs`. Its forward transform maps rectified `(disp, cdisp)` pixel
coordinates to detector coordinates, with the cross-dispersion coordinate passing through
unchanged, and the inverse is attached when the image shape is known. That makes the solution
usable **outside specreduce**: any GWCS-aware code, or any ASDF reader in the case of the file,
can evaluate the transformation without importing this package, whether to extract along curves
of constant wavelength on the unrectified frame, to forward-model the 2D image, or to read
a calibration that may outlive the code that produced it. One detail matters when doing so: the
file carries the image shape as the GWCS **bounding box**, while the live `gwcs` property is left
unbounded, because a bounded GWCS returns `NaN` outside its box instead of extrapolating and would
silently blank the frame edges.

## Summary

- Optical distortion makes wavelength vary along the slit. On these frames a single arc line
  drifts **5.6 pixels** between the ends of the illuminated slit, against a line width of 2.5
  pixels; after rectification the drift is under 0.1 pixels.
- `TiltCorrection` finds arc lines at several rows, subtracting each row's baseline first, fits a
  2D polynomial from rectified space to detector space, and `solution.resample` rebins frames onto
  the corrected grid, conserving flux to better than 0.1 percent. The whole `NDData` comes across:
  flux, propagated variance, mask and metadata. The variance is sub-Poisson by construction and
  cannot express the correlation resampling introduces between neighboring columns.
- Set the reference row near where the science spectrum falls: it defines the wavelength scale
  the rectified frame inherits.
- `degree=4` with `n_cdisp_samples=7` works well here. Keep the samples comfortably above the
  degree, since an underdetermined fit reports an excellent residual and a broken rectification,
  and validate on the rectified frame rather than on the fit statistic. One lamp is enough; more
  buy coverage.
- **Serialize the solution, do not refit it.** `to_asdf` writes it as an analytic GWCS model that
  round-trips bit-for-bit, and `TiltSolution.gwcs` exposes the same transformation to any
  GWCS-aware tool.
- **The step is optional.** Compare the centroid drift along the slit with the instrumental line
  width; if the ratio is well under one, skip it.

Next: [Background Subtraction](07-background.ipynb), which measures the sky from rows either side
of the trace, an operation that only means what it is supposed to mean once the columns are
wavelengths.